<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW13 · Session 1 · Case Study: Import, Audit, Clean, and Explore

        **Course level:** developing beginner  
        **Prior learning:** TW9 statistics, TW10 importing, TW11 cleaning, and TW12 visualisation  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        This case study brings earlier skills together around one business question: which regions and channels generate strong revenue, and what data-quality limitations affect the answer?

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - translate a business question into measurable outputs
- import and audit a supplied retail dataset
- apply a documented cleaning sequence
- perform univariate and grouped exploratory analysis
- separate observed findings from limitations and recommendations


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


## Case-study brief

**Decision:** allocate operational attention across regions and channels.

**Primary measures:** net revenue, transaction count, median transaction value, return rate, and satisfaction.

**Unit of analysis:** one order row after exact duplicate removal.

**Important limitation:** the dataset is synthetic and observational. It can describe associations but cannot prove that a channel or region caused an outcome.


### Import and preserve the source

The file path, shape, and initial sample become the first audit record.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

source_path = DATA_DIR / "business_sales_dirty.csv"
raw = pd.read_csv(source_path, encoding="utf-8-sig")
print("Source:", source_path.name)
print("Shape:", raw.shape)
print(raw.head())


### Define expected schema

A schema contract identifies accidental column changes before analysis.


In [ ]:
expected_columns = {
    "order_id", "order_date", "customer_id", "region", "category",
    "product_name", "channel", "units", "unit_price", "discount",
    "marketing_spend", "satisfaction", "returned",
}
missing_columns = expected_columns - set(raw.columns)
unexpected_columns = set(raw.columns) - expected_columns

print("Missing columns:", sorted(missing_columns))
print("Unexpected columns:", sorted(unexpected_columns))
assert not missing_columns


### Create the baseline audit

The profile is stored before any values change.


In [ ]:
baseline_audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": raw.isna().mean().mul(100).round(1),
    "unique": raw.nunique(dropna=True),
})
print(baseline_audit)
print("Exact duplicates:", raw.duplicated().sum())
print("Duplicate order IDs:", raw.duplicated("order_id").sum())


## Cleaning plan

1. remove exact duplicate rows;
2. standardise region and channel labels;
3. parse dates and isolate failures;
4. mark invalid numerical values as missing;
5. add indicators and apply explicit missing-value treatments;
6. validate the cleaned result;
7. calculate analytical features.


### Execute text, date, and duplicate cleaning

Each operation is visible and ordered.


In [ ]:
clean = raw.drop_duplicates().copy()
clean["region"] = clean["region"].astype("string").str.strip().str.title().fillna("Unknown")
clean["channel"] = (
    clean["channel"]
    .astype("string")
    .str.strip()
    .str.title()
    .replace({"Telephone": "Phone"})
)
clean["order_date"] = pd.to_datetime(
    clean["order_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

print("Rows after duplicate removal:", len(clean))
print("Invalid dates:", clean["order_date"].isna().sum())


### Apply numerical validity and missing-data rules

Indicators retain whether a value required treatment. Invalid dates are dropped because time analysis requires them.


In [ ]:
clean["units_invalid_or_missing"] = (
    clean["units"].isna() | ~clean["units"].between(1, 20)
)
clean["price_invalid_or_missing"] = (
    clean["unit_price"].isna() | (clean["unit_price"] <= 0)
)
clean["units"] = clean["units"].mask(~clean["units"].between(1, 20))
clean["unit_price"] = clean["unit_price"].mask(clean["unit_price"] <= 0)

clean["units"] = clean["units"].fillna(clean.groupby("category")["units"].transform("median"))
clean["unit_price"] = clean["unit_price"].fillna(
    clean.groupby("category")["unit_price"].transform("median")
)
clean["satisfaction"] = clean["satisfaction"].fillna(clean["satisfaction"].median())
clean = clean.dropna(subset=["order_date"]).copy()

print(clean[["units", "unit_price", "satisfaction", "order_date"]].isna().sum())


### Validate the clean table

Assertions make the case study fail loudly if future source data violates the contract.


In [ ]:
assert clean["order_id"].is_unique
assert clean["order_date"].notna().all()
assert clean["units"].between(1, 20).all()
assert (clean["unit_price"] > 0).all()
assert clean["discount"].between(0, 0.5).all()
assert clean["returned"].isin(["Yes", "No"]).all()
print("Clean-data contract passed.")


### Engineer analysis features

Features are calculated only from information available in the row.


In [ ]:
clean["gross_revenue"] = clean["units"] * clean["unit_price"]
clean["net_revenue"] = clean["gross_revenue"] * (1 - clean["discount"])
clean["returned_flag"] = clean["returned"].eq("Yes").astype(int)
clean["order_month"] = clean["order_date"].dt.to_period("M").dt.to_timestamp()
clean["order_quarter"] = clean["order_date"].dt.to_period("Q").astype(str)

print(clean[["gross_revenue", "net_revenue", "returned_flag", "order_month"]].head())


## Exploratory analysis

EDA asks structured questions rather than generating every possible summary. We start with distributions, then compare groups and time.


### Summarise overall performance

Median transaction value is reported alongside total revenue and return rate.


In [ ]:
overall = pd.Series({
    "orders": clean["order_id"].nunique(),
    "customers": clean["customer_id"].nunique(),
    "total_net_revenue": clean["net_revenue"].sum(),
    "median_transaction": clean["net_revenue"].median(),
    "return_rate_pct": clean["returned_flag"].mean() * 100,
    "mean_satisfaction": clean["satisfaction"].mean(),
})
print(overall.round(2))


### Compare regions and channels

Group summaries keep sample size visible and avoid ranking on revenue alone.


In [ ]:
performance = (
    clean.groupby(["region", "channel"], as_index=False)
    .agg(
        orders=("order_id", "count"),
        customers=("customer_id", "nunique"),
        revenue=("net_revenue", "sum"),
        median_transaction=("net_revenue", "median"),
        return_rate=("returned_flag", "mean"),
        satisfaction=("satisfaction", "mean"),
    )
)
performance["return_rate_pct"] = performance["return_rate"] * 100
print(performance.sort_values("revenue", ascending=False).head(10).round(2))


### Visualise revenue and return rate

Two panels separate scale from quality rather than combining incompatible units on dual axes.


In [ ]:
region_summary = clean.groupby("region").agg(
    revenue=("net_revenue", "sum"),
    return_rate=("returned_flag", "mean"),
).sort_values("revenue", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(region_summary.index, region_summary["revenue"], color="#2563eb")
axes[0].set_title("Net revenue")
axes[0].set_ylabel("Revenue (£)")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(region_summary.index, region_summary["return_rate"] * 100, color="#d97706")
axes[1].set_title("Return rate")
axes[1].set_ylabel("Returned orders (%)")
axes[1].tick_params(axis="x", rotation=30)

fig.suptitle("Regional performance has more than one dimension")
fig.tight_layout()
plt.show()
plt.close(fig)


### Check monthly trend

The series aggregates net revenue by month and retains chronological order.


In [ ]:
monthly_revenue = clean.groupby("order_month")["net_revenue"].sum().sort_index()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly_revenue.index, monthly_revenue.values, linewidth=2)
ax.set_title("Monthly net revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (£)")
ax.grid(axis="y", alpha=0.25)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
plt.close(fig)


## Findings, limitations, and recommendations

A responsible conclusion distinguishes:

- **finding:** what the analysed data shows;
- **limitation:** why the estimate may be incomplete or biased;
- **recommendation:** a proportionate next action tied to evidence.

Avoid causal wording because regions and channels were not randomly assigned.


### Generate a reproducible headline table

The top region by revenue is identified programmatically, while quality measures remain available for context.


In [ ]:
regional_headlines = (
    clean.groupby("region", as_index=False)
    .agg(
        orders=("order_id", "count"),
        revenue=("net_revenue", "sum"),
        return_rate=("returned_flag", "mean"),
        satisfaction=("satisfaction", "mean"),
    )
    .sort_values("revenue", ascending=False)
)
top_region = regional_headlines.iloc[0]
print(regional_headlines.round(3))
print(
    f"Highest observed revenue: {top_region['region']} "
    f"with £{top_region['revenue']:,.0f}."
)


## Student coding lab

Treat the five tasks as one mini case study and keep a short decision log.


        ### Student task 1: Write an analysis contract

        <div class="task"><strong>Your job:</strong> Define the business question, unit of analysis, required columns, primary measures, and at least three limitations in Markdown comments or Python strings.</div>

        **Check your work**

        - The question supports a decision.
- The unit is one clean order.
- Limitations distinguish description from causation.


In [ ]:
# STUDENT TASK 1
analysis_contract = {
    "question": "",
    "unit_of_analysis": "",
    "required_columns": [],
    "primary_measures": [],
    "limitations": [],
}
print(analysis_contract)


        ### Student task 2: Create a baseline audit

        <div class="task"><strong>Your job:</strong> Build one table with types, missing count, missing percentage, unique count, and a separate report of duplicates and validity failures.</div>

        **Check your work**

        - The audit is calculated before cleaning.
- Exact and key duplicates are separated.
- Invalid price, units, and date counts are included.


In [ ]:
# STUDENT TASK 2
task_audit = pd.DataFrame(index=raw.columns)
issue_counts = {}
print(task_audit)
print(issue_counts)


        ### Student task 3: Rebuild the cleaning pipeline

        <div class="task"><strong>Your job:</strong> Starting from raw, apply the documented sequence and produce a before/after table for rows, missing cells, duplicates, and validity failures.</div>

        **Check your work**

        - No mutation is made to raw.
- The order of transformations is explicit.
- Final assertions pass.


In [ ]:
# STUDENT TASK 3
task_clean = raw.copy()
before_after = pd.DataFrame()
# Add the cleaning pipeline and quality comparison.
print(before_after)


        ### Student task 4: Answer a grouped question

        <div class="task"><strong>Your job:</strong> Compare categories using orders, total revenue, median transaction, return rate, and satisfaction. Sort and identify one trade-off.</div>

        **Check your work**

        - The clean table is used.
- Rates are formatted clearly.
- The interpretation mentions sample size and a trade-off.


In [ ]:
# STUDENT TASK 4
category_performance = pd.DataFrame()
print(category_performance)
# Interpretation:


        ### Student task 5: Create a two-chart recommendation

        <div class="task"><strong>Your job:</strong> Build two complementary charts supporting one recommendation. Add one finding, one limitation, and one next action as strings.</div>

        **Check your work**

        - Both charts answer the stated question.
- Units and scope are clear.
- Recommendation strength matches observational evidence.



**Optional extension:** Add a sensitivity check using median rather than mean transaction value.


In [ ]:
# STUDENT TASK 5
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# Add two focused plots.
plt.close(fig)
finding = ""
limitation = ""
next_action = ""
print(finding, limitation, next_action)


        ## Knowledge check

        Answer these without running new code first.

        1. Why define a unit of analysis?
2. What should be recorded before cleaning?
3. Why separate revenue and return rate panels?
4. What makes an EDA question structured?
5. How does a recommendation differ from a causal claim?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Session 2 turns the clean case study into reproducible features and a model-ready preprocessing pipeline.
